# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S04: End-to-End Image Classification Pipeline với PyTorch CNN

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S04
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook huấn luyện End-to-End CNN phân loại ảnh hoàn chỉnh
* **Mức độ liên quan VOAI / Olympic AI:** Mô hình thị giác cơ sở tự viết bắt buộc phải làm chủ trước khi dùng mô hình có sẵn.

#### Mục tiêu học tập (Learning Objectives)
- Xây dựng hoàn chỉnh một kiến trúc mạng nơ-ron tích chập (CNN) dạng khối kinh điển: [Conv - BatchNorm - ReLU - MaxPool].
- Làm chủ kỹ thuật làm phẳng không gian (Flatten) hoặc Global Average Pooling để chuyển giao mượt mà sang tầng phân loại (Classifier Head).
- Viết trọn vẹn luồng nạp dữ liệu ảnh, tiền xử lý và vòng lặp huấn luyện CNN trên tập dữ liệu phân loại đa lớp.
- Đánh giá chi tiết hiệu năng mô hình thông qua Confusion Matrix trên từng lớp hình ảnh cụ thể.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành các buổi M4_S01, M4_S02, M4_S03.
- Đã nắm vững Training Loop từ Module M3.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Biết từng tầng `Conv2d` hay `MaxPool2d` đơn lẻ là chưa đủ. Trong một kỳ thi thực tế, nhiệm vụ của bạn là phải lắp ráp các tầng đó thành một **Cỗ máy Phân loại Ảnh hoàn chỉnh (End-to-End Vision Pipeline)**: Từ việc đọc ảnh thô, đưa qua các khối tích chập trích xuất đặc trưng phân cấp, làm phẳng không gian, và đưa ra quyết định phân loại nhãn chính xác.

Buổi học hôm nay đưa bạn bước lên vũ đài thực chiến: Tự tay thiết kế, huấn luyện và đánh giá một mạng CNN tiêu chuẩn Olympic từ số 0, thiết lập nền móng vững chắc trước khi tiến tới các mô hình phức tạp hơn.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Cấu trúc một mạng CNN phân loại ảnh chuẩn mực
Gồm 2 khối lớn:
1. **Feature Extractor (Khối trích xuất đặc trưng không gian):**
   - Xếp chồng các khối lặp lại: `[Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d]`.
   - Càng vào sâu, số kênh đặc trưng (Channels) càng tăng lên ($16 \to 32 \to 64$), trong khi độ phân giải không gian ($H, W$) giảm dần.
2. **Classifier Head (Đầu phân loại):**
   - `nn.Flatten()` duỗi phẳng tensor $3D$ thành vector $1D$.
   - Các tầng `nn.Linear` kết hợp `nn.Dropout` đưa ra vector Logits có kích thước bằng số lượng lớp cần phân loại ($C$).

### 2. Vai trò của Tầng Chuẩn hóa theo Lô (Batch Normalization - `nn.BatchNorm2d`)
* Chuẩn hóa đầu ra của mỗi tầng tích chập về phân phối có trung bình 0 và phương sai 1 trên từng Mini-batch.
* Giúp triệt tiêu hiện tượng dịch chuyển hiệp biến nội tại (Internal Covariate Shift), cho phép sử dụng tốc độ học (Learning Rate) lớn hơn và huấn luyện mô hình nhanh gấp nhiều lần!



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Kiến trúc End-to-End CNN Pipeline:
[Ảnh đầu vào 28x28] 
         │
         ▼
[Khối Conv 1: 1 -> 16 channels, Pool -> 14x14]
         │
         ▼
[Khối Conv 2: 16 -> 32 channels, Pool -> 7x7]
         │
         ▼
[Flatten Layer] ──> Vector 1D (32 * 7 * 7 = 1568 phần tử)
         │
         ▼
[Fully Connected Classifier: Linear(1568, 64) -> Dropout -> Linear(64, num_classes)] ──> [Logits]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 1. Thiet ke kien truc CNN hoan chinh bang nn.Module
class MiniVisionCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(MiniVisionCNN, self).__init__()
        
        # Khoi trich xuat dac trung (Feature Extractor)
        self.features = nn.Sequential(
            # Block 1: 1 channel in -> 16 channels out
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 28x28 -> 14x14
            
            # Block 2: 16 channels in -> 32 channels out
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 14x14 -> 7x7
        )
        
        # Dau phan loai (Classifier Head)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes) # Tra ve Logits cho 3 lop
        )
        
    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits

# Khoi tao mo hinh
model_cnn = MiniVisionCNN(num_classes=3)
print("===== KIEN TRUC MANG CNN HOAN CHINH =====")
print(model_cnn)

### 🔍 Code Walkthrough (Khối 1)
Khối code định nghĩa kiến trúc CNN chuẩn công nghiệp:
- 2 khối tích chập chuẩn hóa đầy đủ với `BatchNorm2d`.
- Kết nối tự động qua `nn.Flatten()`.
- Tầng phân loại có Dropout chống quá khớp, sẵn sàng phân loại 3 lớp hình ảnh.


In [ ]:
# 2. Huan luyen va Danh gia tren Tap du lieu anh nhan tao mo phong 3 lop
torch.manual_seed(42)

# Tao 300 anh nhan tao (100 anh moi lop, kich thuoc 1x28x28)
X_synth = torch.randn(300, 1, 28, 28)
y_synth = torch.tensor([0]*100 + [1]*100 + [2]*100, dtype=torch.long)

# Tao DataLoader
dataset = TensorDataset(X_synth, y_synth)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.005)

# Huan luyen nhanh 5 epochs
model_cnn.train()
for epoch in range(1, 6):
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_cnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        
    epoch_loss = running_loss / len(dataset)
    print(f"Epoch {epoch}/5 | Training CrossEntropy Loss: {epoch_loss:.4f}")

# 3. Danh gia va Xuat Confusion Matrix
model_cnn.eval()
with torch.no_grad():
    test_logits = model_cnn(X_synth)
    test_preds = torch.argmax(test_logits, dim=1).numpy()

cm = confusion_matrix(y_synth.numpy(), test_preds)
print("\n===== MA TRAN NHAM LAN PHAN LOAI ANH (CONFUSION MATRIX) =====")
print(cm)

### 🔍 Code Walkthrough (Khối 2)
Khối code thực thi trọn vẹn luồng huấn luyện và đánh giá:
- Sử dụng hàm mất mát chuẩn mực cho phân loại đa lớp: `nn.CrossEntropyLoss()`.
- Trích xuất nhãn dự đoán bằng `torch.argmax(logits, dim=1)`.
- Xuất ma trận nhầm lẫn thể hiện rõ năng lực phân loại trên từng lớp.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Thử nghiệm với Global Average Pooling
Thay thế toàn bộ cụm `[nn.Flatten(), nn.Linear(32 * 7 * 7, 64)]` bằng một tầng `nn.AdaptiveAvgPool2d((1, 1))` kết hợp `nn.Flatten()`. Nêu rõ số lượng tham số giảm đi bao nhiêu lần khi áp dụng kỹ thuật này.

### Bài 2 (Easy): Mở rộng số lượng khối Tích chập
Thêm một khối tích chập thứ 3 vào mạng: đưa từ 32 channels lên 64 channels. Tính toán kích thước không gian còn lại sau 3 lần MaxPool từ ảnh gốc $28 	imes 28$.

### Bài 3 (Medium): Thêm hàm tính Accuracy trực tiếp trong Training Loop
Viết lại vòng lặp huấn luyện sao cho ở cuối mỗi epoch, chương trình tự động in ra cả hai chỉ số: `Epoch Loss` và `Epoch Accuracy (%)`.

### Bài 4 (Medium): Phân tích Confusion Matrix
Nhìn vào kết quả ma trận nhầm lẫn ở code cell trên, hãy chỉ ra lớp nào đang bị mô hình dự đoán nhầm nhiều nhất sang lớp nào và đưa ra nhận xét.

### Bài 5 (Challenge): Lưu lại các mẫu ảnh bị phân loại sai (Misclassified Visualizer)
Viết một hàm `visualize_errors(images, true_labels, pred_labels, n=4)` lọc ra các bức ảnh mà mô hình đoán sai, sau đó dùng Matplotlib vẽ các bức ảnh đó lên màn hình kèm theo tiêu đề: *"Thực tế: Lớp A - Đoán nhầm: Lớp B"*.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong phần thi Computer Vision của Olympic AI, thời gian huấn luyện mô hình luôn bị ban tổ chức khống chế nghiêm ngặt. Việc thiết kế một mạng CNN gọn nhẹ nhưng hiệu quả (dưới 500,000 tham số) kết hợp đúng đắn với `BatchNorm` sẽ giúp mô hình của bạn hội tụ chỉ sau 10-15 phút, bỏ xa những đối thủ cố chấp sử dụng các mô hình quá cồng kềnh dẫn tới không kịp nộp bài trước giờ đóng cổng.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh tính toán sai kích thước đầu vào của tầng Linear sau khi Flatten:

```python
# CODE CÓ LỖI TÍNH SAI CHIỀU FLATTEN:
# Sau 2 lan MaxPool, anh 28x28 bien thanh 7x7 voi 32 channels.
# Tong so phan tu phai la 32 * 7 * 7 = 1568.
# Neu thi sinh viet: nn.Linear(32 * 14 * 14, 64) -> PyTorch se bao loi Shape Mismatch ngay lap tuc!
```

**Nhiệm vụ của bạn:**
1. Trình bày công thức tính số lượng phần tử phẳng: $N_{flat} = C_{out} 	imes H_{out} 	imes W_{out}$.
2. Nêu mẹo debug nhanh: Đặt một lệnh `print(x.shape)` tạm thời trong hàm `forward` để xem kích thước thực tế trước khi đi vào tầng Linear.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Hàm mất mát nào sau đây là lựa chọn CHUẨN MỰC trong PyTorch cho bài toán phân loại hình ảnh đa lớp (Multi-class Image Classification)?
   - A. `nn.MSELoss`
   - B. `nn.CrossEntropyLoss`
   - C. `nn.L1Loss`
   - D. `nn.CosineSimilarity`

2. Tầng `nn.BatchNorm2d` mang lại lợi ích quan trọng nhất nào cho quá trình huấn luyện mạng CNN?
   - A. Giảm kích thước ảnh đi 4 lần.
   - B. Ổn định phân phối dữ liệu qua từng tầng, giúp quá trình huấn luyện hội tụ nhanh hơn và cho phép sử dụng Learning Rate cao hơn.
   - C. Chuyển ảnh màu thành ảnh xám.
   - D. Tự động gán nhãn cho ảnh.

3. Để lấy ra lớp có xác suất dự đoán cao nhất từ tensor Logits đầu ra của mạng, ta sử dụng hàm nào trong PyTorch?
   - A. `torch.max_val()`
   - B. `torch.argmax(logits, dim=1)`
   - C. `torch.sum()`
   - D. `torch.sort()`

4. Nếu ảnh đầu vào có kích thước $32 	imes 32$, sau khi đi qua 2 tầng `nn.MaxPool2d(2, 2)` liên tiếp, kích thước không gian còn lại là bao nhiêu?
   - A. $16 	imes 16$
   - B. $8 	imes 8$
   - C. $4 	imes 4$
   - D. $2 	imes 2$

5. Lớp `nn.Flatten()` trong kiến trúc CNN thực hiện thao tác gì?
   - A. Xóa bỏ hoàn toàn các giá trị pixel bằng 0.
   - B. Duỗi phẳng tensor đặc trưng đa chiều $[Batch, Channels, H, W]$ thành vector 2 chiều $[Batch, Channels 	imes H 	imes W]$ để sẵn sàng đưa vào các tầng Fully Connected.
   - C. Làm mờ bức ảnh.
   - D. Đổi thứ tự kênh màu.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Cấu trúc 2 khối hoàn chỉnh của mạng phân loại ảnh: Feature Extractor và Classifier Head.
  - Sức mạnh của tầng `nn.BatchNorm2d` trong việc tăng tốc hội tụ.
  - Cách tính toán chính xác kích thước duỗi phẳng (Flatten dimension).
  - Đánh giá toàn diện mô hình thị giác bằng Confusion Matrix.
* **What I can now do:**
  - Tự tay lập trình và huấn luyện một mạng CNN hoàn chỉnh từ đầu đến cuối trên PyTorch.
  - Tự tin debug mọi lỗi lệch kích thước tầng tích chập.
* **What I still need to practice:**
  - Khám phá bí quyết đạt điểm cao trong thi đấu: **Transfer Learning** ở Buổi 5.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S05**, chúng ta sẽ làm quen với vũ khí tối thượng của các vận động viên AI: **Transfer Learning Trong Thị Giác Máy Tính: Pretrained Models & Error Analysis** — Học cách đứng trên vai người khổng lồ (ResNet, MobileNet), đóng băng trọng số (Freeze) và phân tích các ca dự đoán sai lầm!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Kiến trúc End-to-End CNN ([Conv-BN-ReLU-Pool] blocks) ✓
- Flatten layer & Fully connected classifier head ✓
- Training loop cho phân loại ảnh & Confusion matrix evaluation ✓

* **Session Deliverable:** Mã nguồn End-to-End Image Classification Pipeline hoàn chỉnh bằng PyTorch CNN
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
